# FoodBridge Analytics — Machine Learning Data Preparation

**Purpose:** turn the cleaned FoodBridge tables into one analysis-ready table for the demand model: **one row per Second Harvest partner organization (912 rows)**.

**What this notebook does**
1. Loads the cleaned data exported by the Assignment 2 cleaning notebook (`FoodBridge_Cleaned_Data.xlsx`).
2. Builds one row per partner (profile + demand).
3. Adds supply totals as reference columns (not model inputs).
4. Prepares two targets: **Users Served in 2025** (main) and **Unmet Demand** (second).
5. Groups text categories, turns food types and flags into 0/1 columns.
6. Adds provincial context from Statistics Canada (food insecurity 2024, population 2025).
7. Applies missing-value rules, labels each column's role, validates and exports.

**Not done here (left for the machine learning stage):** one-hot encoding of categories and the train/test split.

All Second Harvest activity values are synthetic/mock and must not be presented as verified real-world activity.

## 1. Setup and load the cleaned data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 60); pd.set_option('display.max_colwidth', 80)

CLEANED_FILE = 'FoodBridge_Cleaned_Data.xlsx'
try:  # Google Colab: upload the cleaned workbook if it is not already present
    from google.colab import files
    if not Path(CLEANED_FILE).exists():
        files.upload()
except ImportError:
    pass

candidates = [Path(CLEANED_FILE), Path('foodbridge_outputs') / CLEANED_FILE]
CLEANED_PATH = next((p for p in candidates if p.exists()), None)
if CLEANED_PATH is None:
    raise FileNotFoundError(f'Upload {CLEANED_FILE} (output of the Assignment 2 cleaning notebook).')

sheets = pd.read_excel(CLEANED_PATH, sheet_name=None)
org, pop = sheets['SH Organizations'], sheets['Population Demand']
alloc, ngo = sheets['Donation Allocations'], sheets['National NGO']

EXPECTED_ROWS = {'SH Organizations': 912, 'Population Demand': 912, 'Donation Allocations': 4805,
                 'National NGO': 11286, 'Food Donors': 227, 'Food Donations': 1200}
load_check = pd.DataFrame([{'Sheet': s, 'Rows loaded': len(sheets[s]), 'Rows expected': n,
                            'Status': 'Pass' if len(sheets[s]) == n else 'Review'} for s, n in EXPECTED_ROWS.items()])
display(load_check)

,Sheet,Rows loaded,Rows expected,Status
0,SH Organizations,912,912,Pass
1,Population Demand,912,912,Pass
2,Donation Allocations,4805,4805,Pass
3,National NGO,11286,11286,Pass
4,Food Donors,227,227,Pass
5,Food Donations,1200,1200,Pass


## 2. One row per partner organization

SH Organizations (profile) is joined to Population Demand (demand fields) on `Organization ID`. Both tables have 912 unique IDs, so the join must keep exactly 912 rows.

In [ ]:
PROVINCE_FIELD_DROP = ['Province/Territory', 'Postal Code', 'Postal Code Invalid']   # duplicated in SH Organizations
partners = org.merge(pop.drop(columns=PROVINCE_FIELD_DROP), on='Organization ID', how='inner', validate='one_to_one')
assert len(partners) == 912 and partners['Organization ID'].is_unique
print('Partner table:', partners.shape)

Partner table: (912, 26)


## 3. Supply totals (reference only)

Each partner's received meals, pounds, number of donations and distinct food types are summed from the allocation table. These columns are **reference only**: meals were created from users served, so using them as model inputs would give the model the answer. They are kept to compare supply with estimated demand after modelling and for the dashboard.

In [ ]:
supply = (alloc.groupby('Recipient Organization ID')
               .agg(supply_meals_received=('Allocated Estimated Meals', 'sum'),
                    supply_pounds_received=('Allocated Quantity in Pounds', 'sum'),
                    supply_donations_received=('Donation ID', 'nunique'),
                    supply_food_types_received=('Food Type', 'nunique'))
               .rename_axis('Organization ID').reset_index())
partners = partners.merge(supply, on='Organization ID', how='left', validate='one_to_one')
supply_cols = ['supply_meals_received', 'supply_pounds_received', 'supply_donations_received', 'supply_food_types_received']
partners[supply_cols] = partners[supply_cols].fillna(0).astype(int)

meal_diff = (partners['supply_meals_received'] - partners['Meals Provided by Second Harvest (Synthetic Estimate)']).abs()
print('Partners whose allocated meals differ from the demand file:', int((meal_diff > 0).sum()))
partners['supply_meals_per_user_served'] = (partners['supply_meals_received'] / partners['Users Served in 2025']).round(2)
display(partners[['Organization ID'] + supply_cols + ['supply_meals_per_user_served']].describe().round(1))

Partners whose allocated meals differ from the demand file: 0


,supply_meals_received,supply_pounds_received,supply_donations_received,supply_food_types_received,supply_meals_per_user_served
count,912.0,912.0,912.0,912.0,911.0
mean,105900.5,105900.5,5.3,3.4,167.8
std,392570.1,392570.1,6.4,2.1,125.0
min,29.0,29.0,1.0,1.0,3.6
25%,2581.8,2581.8,2.0,2.0,78.9
50%,6977.5,6977.5,4.0,3.0,129.4
75%,19769.5,19769.5,5.0,4.0,252.0
max,5962414.0,5962414.0,50.0,10.0,445.8


## 4. Targets

- **Main target:** `target_users_served` = Users Served in 2025.
- **Second target:** `target_unmet_demand` = Unmet Demand (estimated people).
- Both are skewed (from a handful of people to almost 18,000), so a log version `log(1 + value)` is added for modelling.
- `has_users_target` / `has_unmet_target` mark rows that can be used to train each model (missing targets are never set to zero).
- `users_served_source` (Original estimate / Randomized) is kept so the team can choose which rows to train on later.

In [ ]:
partners['target_users_served'] = partners['Users Served in 2025']
partners['target_unmet_demand'] = partners['Unmet Demand (Estimated People; Synthetic)']
partners['log_target_users_served'] = np.log1p(partners['target_users_served'])
partners['log_target_unmet_demand'] = np.log1p(partners['target_unmet_demand'])
partners['has_users_target'] = partners['target_users_served'].notna().astype(int)
partners['has_unmet_target'] = partners['target_unmet_demand'].notna().astype(int)
partners['users_served_source'] = partners['Users Served Source']

target_summary = pd.DataFrame({
    'Target': ['Users served (main)', 'Unmet demand (second)'],
    'Rows with target': [partners['has_users_target'].sum(), partners['has_unmet_target'].sum()],
    'Rows without target': [(1 - partners['has_users_target']).sum(), (1 - partners['has_unmet_target']).sum()],
    'Min': [partners['target_users_served'].min(), partners['target_unmet_demand'].min()],
    'Median': [partners['target_users_served'].median(), partners['target_unmet_demand'].median()],
    'Max': [partners['target_users_served'].max(), partners['target_unmet_demand'].max()]})
display(target_summary)
display(pd.crosstab(partners['users_served_source'], partners['has_users_target'].map({1: 'Has target', 0: 'Missing'})))

,Target,Rows with target,Rows without target,Min,Median,Max
0,Users served (main),911,1,2.0,54.0,17940.0
1,Unmet demand (second),761,151,0.0,10.0,6106.0


has_users_target,Has target,Missing
users_served_source,,
Original estimate,100,1
Randomized,811,0


## 5. Group text categories

Too many small categories make patterns hard to learn. Each field is mapped to a few groups with keyword rules (applied in order). The mapping tables below show every original value and its group so the rules can be reviewed.

In [ ]:
def first_match(text, rules, default):
    t = str(text).casefold()
    for group, keywords in rules:
        if any(k in t for k in keywords):
            return group
    return default

ORG_RULES = [
    ('Food bank / food recovery', ['food bank', 'food recovery', 'food redistribution']),
    ('Community / social services', ['community food centre', 'social services', 'community support', 'community health centre']),
    ('Education', ['educational facility', 'university', 'college', 'school board']),
    ('Health care', ['healthcare facility', 'hospital']),
    ('Shelter / housing', ['shelter', 'transitional housing']),
    ('Faith-based', ['faith-based']),
    ('Registered charity', ['registered charity']),
]
SERVICE_RULES = [
    ('Not verified', ['not verified; potential community partner only']),
    ('Food assistance / recovery', ['food bank', 'food assistance', 'food recovery', 'food rescue', 'food collection',
                                    'food distribution', 'community meals', 'community kitchen', 'food access']),
    ('Education', ['education']),
    ('Health', ['hospital', 'health']),
    ('Residential / shelter', ['resident', 'shelter']),
]
TARGET_POP_FLAGS = {   # flag column -> keyword in Target Populations
    'serves_children_youth': 'children/youth', 'serves_general_food_need': 'people and households needing food assistance',
    'serves_families': 'families/households', 'serves_seniors': 'seniors',
    'serves_housing_insecure': 'homelessness', 'serves_newcomers': 'newcomers', 'serves_indigenous': 'indigenous'}

partners['org_group'] = partners['Organization type'].map(lambda v: first_match(v, ORG_RULES, 'Other'))
partners['service_group'] = partners['Food-related service or program'].map(lambda v: first_match(v, SERVICE_RULES, 'Community / other'))
tp = partners['Target Populations'].astype(str).str.casefold()
for col, key in TARGET_POP_FLAGS.items():
    partners[col] = tp.str.contains(key, regex=False).astype(int)
n_groups = partners[list(TARGET_POP_FLAGS)].sum(axis=1)
single = {'serves_children_youth': 'Children / youth', 'serves_general_food_need': 'General food need', 'serves_families': 'Families'}
def target_group(row, n):
    if n == 0: return 'Not documented'
    if n > 1: return 'Multiple groups'
    flag = next(c for c in TARGET_POP_FLAGS if row[c] == 1)
    return single.get(flag, 'Other specific group')   # seniors, housing insecure, newcomers, Indigenous (small groups)
partners['target_pop_group'] = [target_group(r, n) for (_, r), n in zip(partners.iterrows(), n_groups)]

grouping_maps = {
    'Organization type': partners.groupby(['org_group', 'Organization type']).size().rename('Partners').reset_index(),
    'Service': partners.groupby(['service_group', 'Food-related service or program']).size().rename('Partners').reset_index(),
    'Target population': partners.groupby(['target_pop_group', 'Target Populations']).size().rename('Partners').reset_index()}
for name, col in [('Organization type', 'org_group'), ('Service', 'service_group'), ('Target population', 'target_pop_group')]:
    print(f'{name}: {grouping_maps[name].iloc[:, 1].nunique()} original values -> {partners[col].nunique()} groups')
    display(partners[col].value_counts().rename('Partners').to_frame())

Organization type: 184 original values -> 8 groups


,Partners
org_group,
Registered charity,525
Education,212
Health care,75
Food bank / food recovery,71
Faith-based,13
Shelter / housing,9
Community / social services,6
Other,1


Service: 33 original values -> 6 groups


,Partners
service_group,
Not verified,769
Food assistance / recovery,74
Education,31
Community / other,17
Health,12
Residential / shelter,9


Target population: 11 original values -> 6 groups


,Partners
target_pop_group,
Not documented,537
Children / youth,238
General food need,75
Multiple groups,23
Other specific group,23
Families,16


## 6. Food types accepted → numeric features

The text list in `Food Types Accepted` becomes a count plus one 0/1 column per food type.

In [ ]:
FOOD_TYPES = ['Bakery', 'Beverages', 'Dairy and eggs', 'Fresh produce', 'Frozen foods', 'Meat and poultry',
              'Mixed food products', 'Pantry staples', 'Prepared foods', 'Snacks and confectionery']
accepted = partners['Food Types Accepted'].fillna('').str.split(';').map(lambda xs: {x.strip() for x in xs if x.strip()})
unknown_types = set().union(*accepted) - set(FOOD_TYPES)
print('Food types not in the standard list:', unknown_types or 'none')
FOOD_FLAG_COLS = []
for ft in FOOD_TYPES:
    col = 'accepts_' + ft.lower().replace(' and ', '_').replace(' ', '_')
    partners[col] = accepted.map(lambda s: int(ft in s)); FOOD_FLAG_COLS.append(col)
partners['n_food_types_accepted'] = accepted.map(len)
display(partners[FOOD_FLAG_COLS + ['n_food_types_accepted']].agg(['sum', 'mean']).round(2).T)

Food types not in the standard list: none


,sum,mean
accepts_bakery,912.0,1.00
accepts_beverages,912.0,1.00
accepts_dairy_eggs,647.0,0.71
accepts_fresh_produce,912.0,1.00
accepts_frozen_foods,591.0,0.65
accepts_meat_poultry,615.0,0.67
accepts_mixed_food_products,912.0,1.00
accepts_pantry_staples,912.0,1.00
accepts_prepared_foods,586.0,0.64
accepts_snacks_confectionery,655.0,0.72


## 7. Flags → 0/1 columns

Models work with numbers, so Yes/No-style fields become 0/1. "Unverified" is kept as its own flag instead of being guessed.

In [ ]:
partners['is_urban'] = partners['Rural/urban classification'].eq('Urban').astype(int)
partners['rural_urban_unverified'] = partners['Rural/urban classification'].eq('Unverified').astype(int)
partners['operating_verified'] = partners['Operating status (simulated)'].eq('Operating').astype(int)
partners['has_website'] = partners['Official website'].astype(str).str.startswith('http').astype(int)
partners['has_coordinates'] = partners['Has Coordinates'].eq('Yes').astype(int)
partners['latitude'] = partners['Latitude']; partners['longitude'] = partners['Longitude']
partners['province'] = partners['Province or territory']
FLAG_COLS = ['is_urban', 'rural_urban_unverified', 'operating_verified', 'has_website', 'has_coordinates']
display(partners[FLAG_COLS].agg(['sum', 'mean']).round(3).T)

KeyError: 'Province or territory'

## 8. Provincial context (Statistics Canada)

Adds a measure of need outside the organization itself.

- **Food insecurity, 2024** (percentage of persons): Statistics Canada, Canadian Income Survey, [Table 13-10-0835-01](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1310083501) (All persons). The latest year available is 2024. `prov_fi_data_quality` is the Statistics Canada quality letter (A = excellent … E = use with caution). The territories' estimates are less reliable.
- **Population, July 1 2025**: Statistics Canada, [Table 17-10-0009-01](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1710000901).
- `prov_food_insecure_people_est` = food-insecure % × population (approximate, because the survey year and population year differ by one year).

In [ ]:
PROVINCIAL_CONTEXT = pd.DataFrame([
    # code, food insecure %, moderate or severe %, severe %, quality, population (2025-07-01)
    ('NL', 25.5, 20.4,  7.5, 'C',    550202), ('PE', 23.3, 18.9,  5.8, 'D',    181594),
    ('NS', 26.2, 19.7,  7.5, 'C',   1095157), ('NB', 28.2, 20.9,  6.8, 'C',    874711),
    ('QC', 18.0, 12.1,  4.1, 'C',   9051022), ('ON', 25.4, 20.2,  7.1, 'B',  16216961),
    ('MB', 27.9, 22.3,  7.0, 'C',   1508582), ('SK', 26.7, 19.7,  6.6, 'C',   1266289),
    ('AB', 28.4, 22.8,  9.4, 'C',   5024631), ('BC', 23.2, 17.4,  5.8, 'C',   5702544),
    ('YT', 15.5, 13.0,  5.6, 'D',     49150), ('NT', 16.4, 12.6,  5.5, 'E',     46031),
    ('NU', 56.4, 53.2, 31.0, 'C',     42108),
], columns=['province', 'prov_food_insecure_pct_2024', 'prov_mod_severe_food_insecure_pct_2024',
            'prov_severe_food_insecure_pct_2024', 'prov_fi_data_quality', 'prov_population_2025'])
PROVINCIAL_CONTEXT['prov_food_insecure_people_est'] = (
    PROVINCIAL_CONTEXT['prov_food_insecure_pct_2024'] / 100 * PROVINCIAL_CONTEXT['prov_population_2025']).round().astype(int)
PROVINCIAL_CONTEXT['prov_sh_partner_count'] = PROVINCIAL_CONTEXT['province'].map(partners['province'].value_counts()).fillna(0).astype(int)
CONTEXT_SOURCES = {
    'Food insecurity (2024)': 'Statistics Canada. Table 13-10-0835-01 Food insecurity by selected demographic characteristics. https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1310083501',
    'Population (2025-07-01)': 'Statistics Canada. Table 17-10-0009-01 Population estimates, quarterly. https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1710000901'}

partners = partners.merge(PROVINCIAL_CONTEXT.drop(columns='prov_sh_partner_count'), on='province', how='left', validate='many_to_one')
print('Partners without provincial context:', int(partners['prov_food_insecure_pct_2024'].isna().sum()))
display(PROVINCIAL_CONTEXT)

KeyError: 'province'

## 9. Missing-value rules for modelling

The cleaning-stage rules are applied in a form the model can use:

| Situation | Rule |
|---|---|
| Users served missing (1 row) | Row kept; `has_users_target = 0` (left out of training for the main model) |
| Unmet demand unknown (151 rows) | Row kept; `has_unmet_target = 0` (left out of training for the second model only) |
| "Unverified" / "Requires verification" | Kept as their own values or flags; not guessed |
| Missing coordinates | Latitude/longitude stay missing; `has_coordinates = 0`. How to fill them is decided at the ML stage |
| Missing text categories | Filled with "Unknown" |

In [ ]:
CATEGORY_INPUTS = ['province', 'org_group', 'service_group', 'target_pop_group']
for col in CATEGORY_INPUTS:
    partners[col] = partners[col].fillna('Unknown')
missing_summary = partners[CATEGORY_INPUTS + ['latitude', 'longitude', 'target_users_served', 'target_unmet_demand']].isna().sum().rename('Missing').to_frame()
display(missing_summary)

## 10. Column roles and the ML dataset

Every column gets one role:

- **ID**: identifies the row.
- **Target**: what the model predicts.
- **Training filter**: chooses which rows to train on.
- **Input**: what the model learns from.
- **Supply reference**: compared with predictions later; never an input.
- **Reference**: readable context for the dashboard and review.

Raw text, names, websites, addresses, notes and the unmet-demand rate are **excluded**. The rate was used to create unmet demand, so it would leak the answer.

In [ ]:
ROLES = {
    'ID': ['organization_id'],
    'Reference': ['organization_name', 'organization_type_raw', 'service_raw', 'target_populations_raw', 'rural_urban_raw'],
    'Target': ['target_users_served', 'log_target_users_served', 'target_unmet_demand', 'log_target_unmet_demand'],
    'Training filter': ['has_users_target', 'has_unmet_target', 'users_served_source'],
    'Input (category)': CATEGORY_INPUTS,
    'Input (numeric)': FLAG_COLS + ['latitude', 'longitude', 'n_food_types_accepted'] + FOOD_FLAG_COLS + list(TARGET_POP_FLAGS)
                       + ['prov_food_insecure_pct_2024', 'prov_mod_severe_food_insecure_pct_2024', 'prov_severe_food_insecure_pct_2024',
                          'prov_population_2025', 'prov_food_insecure_people_est'],
    'Supply reference': supply_cols + ['supply_meals_per_user_served'],
    'Context reference': ['prov_fi_data_quality'],
}
partners = partners.rename(columns={'Organization ID': 'organization_id', 'Organization name': 'organization_name',
                                    'Organization type': 'organization_type_raw', 'Food-related service or program': 'service_raw',
                                    'Target Populations': 'target_populations_raw', 'Rural/urban classification': 'rural_urban_raw'})
ml_columns = [c for cols in ROLES.values() for c in cols]
ml_dataset = partners[ml_columns].copy()
EXCLUDED = sorted(set(partners.columns) - set(ml_columns))
INPUT_COLS = ROLES['Input (category)'] + ROLES['Input (numeric)']
print('ML dataset:', ml_dataset.shape, '| inputs:', len(INPUT_COLS), '| excluded source columns:', len(EXCLUDED))
display(pd.Series({r: len(c) for r, c in ROLES.items()}, name='Columns').to_frame())
display(ml_dataset.head())

## 11. Feature dictionary

In [ ]:
FOOD_DESC = {c: f'1 if the partner accepts {ft.lower()}, else 0.' for c, ft in zip(FOOD_FLAG_COLS, FOOD_TYPES)}
POP_DESC = {c: f'1 if Target Populations mentions "{k}", else 0.' for c, k in TARGET_POP_FLAGS.items()}
DESC = {
 'organization_id': ('Unique partner ID; join key to the cleaned tables.', 'SH Organizations'),
 'organization_name': ('Partner name (reference only).', 'SH Organizations'),
 'organization_type_raw': ('Original organization type before grouping.', 'SH Organizations'),
 'service_raw': ('Original food-related service before grouping.', 'SH Organizations'),
 'target_populations_raw': ('Original target populations before grouping.', 'Population Demand'),
 'rural_urban_raw': ('Original rural/urban value (Urban / Unverified).', 'SH Organizations'),
 'target_users_served': ('Main target: people served in 2025 (synthetic; randomized for most partners).', 'Population Demand'),
 'log_target_users_served': ('log(1 + users served); used for modelling because the target is skewed.', 'Derived'),
 'target_unmet_demand': ('Second target: estimated people whose need is not met (synthetic). Missing = Unknown.', 'Population Demand'),
 'log_target_unmet_demand': ('log(1 + unmet demand).', 'Derived'),
 'has_users_target': ('1 if users served is available (row can train the main model).', 'Derived'),
 'has_unmet_target': ('1 if unmet demand is available (row can train the second model).', 'Derived'),
 'users_served_source': ('Original estimate or Randomized; used to choose training rows later.', 'National NGO via cleaning notebook'),
 'province': ('Province or territory code.', 'SH Organizations'),
 'org_group': ('Organization type grouped into 8 groups (rules in section 5).', 'Derived'),
 'service_group': ('Food-related service grouped into 6 groups.', 'Derived'),
 'target_pop_group': ('Target population grouped: Not documented, Children / youth, General food need, Families, Multiple groups, Other specific group.', 'Derived'),
 'is_urban': ('1 if classified Urban.', 'Derived'), 'rural_urban_unverified': ('1 if rural/urban is Unverified.', 'Derived'),
 'operating_verified': ('1 if operating status is Operating; 0 if it requires verification.', 'Derived'),
 'has_website': ('1 if an official website was found.', 'Derived'),
 'has_coordinates': ('1 if latitude and longitude are available.', 'Derived'),
 'latitude': ('Latitude (missing when has_coordinates = 0).', 'SH Organizations'),
 'longitude': ('Longitude (missing when has_coordinates = 0).', 'SH Organizations'),
 'n_food_types_accepted': ('Number of food types the partner accepts (0-10).', 'Derived'),
 'prov_food_insecure_pct_2024': ('% of persons in food-insecure households in the province, 2024.', 'Statistics Canada Table 13-10-0835-01'),
 'prov_mod_severe_food_insecure_pct_2024': ('% of persons moderately or severely food insecure, 2024.', 'Statistics Canada Table 13-10-0835-01'),
 'prov_severe_food_insecure_pct_2024': ('% of persons severely food insecure, 2024.', 'Statistics Canada Table 13-10-0835-01'),
 'prov_population_2025': ('Provincial population, July 1 2025.', 'Statistics Canada Table 17-10-0009-01'),
 'prov_food_insecure_people_est': ('Estimated food-insecure people in the province (% x population).', 'Derived from Statistics Canada'),
 'prov_fi_data_quality': ('Statistics Canada quality letter for the food-insecurity estimate (A best, E use with caution).', 'Statistics Canada Table 13-10-0835-01'),
 'supply_meals_received': ('Total meals allocated to the partner in 2025 (not a model input).', 'Donation Allocations'),
 'supply_pounds_received': ('Total pounds allocated (not a model input).', 'Donation Allocations'),
 'supply_donations_received': ('Number of donations received (not a model input).', 'Donation Allocations'),
 'supply_food_types_received': ('Number of distinct food types received (not a model input).', 'Donation Allocations'),
 'supply_meals_per_user_served': ('Meals received / users served; for comparing supply with demand.', 'Derived'),
}
REAL = {'organization_name', 'organization_type_raw', 'service_raw', 'province', 'org_group', 'service_group',
        'latitude', 'longitude', 'has_coordinates', 'has_website'}
def origin_of(c):
    if c.startswith('prov_'): return 'Real (Statistics Canada)'
    if c in REAL: return 'Real (public source, approximate coordinates)' if c in ('latitude', 'longitude') else 'Real (public source)'
    if c == 'organization_id': return 'Synthetic ID'
    return 'Synthetic / derived from synthetic'
role_of = {c: r for r, cols in ROLES.items() for c in cols}
feature_dictionary = pd.DataFrame([{
    'Column': c, 'Role': role_of[c],
    'Data type': 'Decimal' if pd.api.types.is_float_dtype(ml_dataset[c]) else 'Integer' if pd.api.types.is_integer_dtype(ml_dataset[c]) else 'Text',
    'Description': (DESC.get(c) or (FOOD_DESC.get(c) or POP_DESC.get(c), 'Derived'))[0],
    'Source': (DESC.get(c) or (None, 'Derived'))[1],
    'Example value': str(ml_dataset[c].dropna().iloc[0]) if ml_dataset[c].notna().any() else '',
    'Missing count': int(ml_dataset[c].isna().sum()),
    'Data origin': origin_of(c)} for c in ml_dataset.columns])
display(feature_dictionary)

## 12. Validation checks

The export runs only if every check passes.

In [ ]:
checks = []
def check(name, rule, failures):
    checks.append({'Check': name, 'Rule': rule, 'Failures': int(failures), 'Status': 'Pass' if failures == 0 else 'Review'})

check('Row count', 'One row per partner (912)', abs(len(ml_dataset) - 912))
check('Unique ID', 'No duplicate organization_id', ml_dataset['organization_id'].duplicated().sum())
check('No supply in inputs', 'No supply_ column is an input', sum(c.startswith('supply_') for c in INPUT_COLS))
check('No target in inputs', 'No target or training-filter column is an input', len(set(INPUT_COLS) & set(ROLES['Target'] + ROLES['Training filter'])))
check('Numeric inputs', 'All numeric inputs are numbers', sum(not pd.api.types.is_numeric_dtype(ml_dataset[c]) for c in ROLES['Input (numeric)']))
check('Input completeness', 'No missing inputs except latitude/longitude',
      ml_dataset[[c for c in INPUT_COLS if c not in ('latitude', 'longitude')]].isna().sum().sum())
check('Coordinates flag', 'latitude missing exactly when has_coordinates = 0',
      (ml_dataset['latitude'].isna() != ml_dataset['has_coordinates'].eq(0)).sum())
check('0/1 flags', 'Flag columns contain only 0 and 1',
      (~ml_dataset[FLAG_COLS + FOOD_FLAG_COLS + list(TARGET_POP_FLAGS) + ['has_users_target', 'has_unmet_target']].isin([0, 1])).sum().sum())
check('Food type count', 'n_food_types_accepted = sum of accepts_ flags', (ml_dataset[FOOD_FLAG_COLS].sum(axis=1) != ml_dataset['n_food_types_accepted']).sum())
check('Target flags', 'has_users_target / has_unmet_target match missing targets',
      (ml_dataset['has_users_target'] != ml_dataset['target_users_served'].notna()).sum() + (ml_dataset['has_unmet_target'] != ml_dataset['target_unmet_demand'].notna()).sum())
check('Unmet <= users', 'Unmet demand never exceeds users served',
      (ml_dataset['target_unmet_demand'] > ml_dataset['target_users_served']).sum())
check('Provincial context', 'Every partner has Statistics Canada context', ml_dataset['prov_food_insecure_pct_2024'].isna().sum())
check('Supply reconciliation', 'Total allocated meals = total meals in the cleaned allocation table',
      int(ml_dataset['supply_meals_received'].sum() != alloc['Allocated Estimated Meals'].sum()))
check('Grouping coverage', 'Every partner has an org, service and target-population group',
      ml_dataset[['org_group', 'service_group', 'target_pop_group']].isna().sum().sum())

validation = pd.DataFrame(checks)
all_pass = bool(validation['Status'].eq('Pass').all())
display(validation)
print('All checks passed.' if all_pass else 'Review the checks marked "Review" before exporting.')

## 13. Export

- `FoodBridge_ML_Dataset.csv`: the model-ready table (one row per partner; categories not yet encoded, no split yet).
- `FoodBridge_ML_Feature_Dictionary.csv`: each column's role, type, description and source.
- `FoodBridge_ML_Prep_Log.xlsx`: load check, target summary, grouping maps, provincial context with sources, missing values and validation.

In [ ]:
if not all_pass:
    raise RuntimeError('Export stopped: review the checks marked "Review".')
OUT = Path('foodbridge_ml_outputs'); OUT.mkdir(exist_ok=True)
dataset_path = OUT / 'FoodBridge_ML_Dataset.csv'
dictionary_path = OUT / 'FoodBridge_ML_Feature_Dictionary.csv'
log_path = OUT / 'FoodBridge_ML_Prep_Log.xlsx'
ml_dataset.to_csv(dataset_path, index=False, encoding='utf-8-sig')
feature_dictionary.to_csv(dictionary_path, index=False, encoding='utf-8-sig')
with pd.ExcelWriter(log_path, engine='openpyxl') as w:
    load_check.to_excel(w, sheet_name='Load Check', index=False)
    target_summary.to_excel(w, sheet_name='Targets', index=False)
    for name, frame in grouping_maps.items():
        frame.to_excel(w, sheet_name=f'Groups - {name}'[:31], index=False)
    PROVINCIAL_CONTEXT.to_excel(w, sheet_name='Provincial Context', index=False)
    pd.Series(CONTEXT_SOURCES, name='Source').rename_axis('Data').reset_index().to_excel(w, sheet_name='Context Sources', index=False)
    missing_summary.reset_index(names='Column').to_excel(w, sheet_name='Missing Values', index=False)
    pd.Series(EXCLUDED, name='Excluded source column').to_frame().to_excel(w, sheet_name='Excluded Columns', index=False)
    validation.to_excel(w, sheet_name='Validation', index=False)
display(pd.DataFrame([{'File': p.name, 'Size (KB)': round(p.stat().st_size / 1024, 1)} for p in [dataset_path, dictionary_path, log_path]]))
try:
    from google.colab import files
    for p in [dataset_path, dictionary_path, log_path]:
        files.download(str(p))
except ImportError:
    print('Files saved to', OUT.resolve())

## Notes for the machine learning stage

- **Still to do at ML time:** one-hot encode the 4 category inputs, handle missing coordinates, and create the train/test split (fixed seed, split by organization).
- **Training rows:** main model uses `has_users_target = 1`; second model uses `has_unmet_target = 1`. Decide whether to keep `users_served_source = Randomized` rows (811 partners) or retrain on original estimates or regenerated values.
- **Constant columns:** `accepts_bakery`, `accepts_beverages`, `accepts_fresh_produce`, `accepts_mixed_food_products` and `accepts_pantry_staples` are 1 for every partner, so they carry no information. Drop them at the ML stage (kept here for completeness).
- **Never use `supply_` columns as inputs.** They are for comparing estimated demand with what partners received.
- Provincial context is real Statistics Canada data; partner activity values are synthetic.